<a href="https://colab.research.google.com/github/muhammadyusuf-pro/Machine-Learning-Course/blob/main/XGBoosting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
from xgboost import XGBClassifier
import pandas as pd
from zipfile import ZipFile
from io import BytesIO
from urllib.request import urlopen
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

URL = ('https://archive.ics.uci.edu/static/public/222/bank+marketing.zip')

outer = ZipFile(BytesIO(urlopen(URL).read()))
inner = ZipFile(BytesIO(outer.read('bank-additional.zip')))
df = pd.read_csv(
    inner.open('bank-additional/bank-additional.csv'),sep=";"
)
df.to_csv('bank.csv',sep=";", index=False)

df.head()

y = df["y"].map({"no": 0, "yes": 1})
X = df.drop(columns=["y"])

# Matnli ustunlarni raqamga aylantirdik
X = pd.get_dummies(X, drop_first=True)

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=42)

def train(lr, Xtr=X_train, ytr=y_train, Xva=X_valid, yva=y_valid, patience=50):
  model = XGBClassifier(n_estimators = 2000, learning_rate = lr, max_depth = 3, early_stopping_rounds=patience, eval_metric = "logloss", random_state=42)
  model.fit(Xtr, ytr, eval_set = [(Xva, yva)], verbose = False)

  pred = model.predict(X_test)
  print("Accuracy: ", round(accuracy_score(y_test, pred), 4))
  print("Best iteration: ", model.best_iteration)
  print("Best score: ", model.best_score)
  return model

model = train(0.05)

Accuracy:  0.9102
Best iteration:  103
Best score:  0.18091928488103046
